# Sahyog Setu — translation into 22 Indian languages
AI4Bharat **IndicTrans2** (open, MIT licence). Runs on Colab's free GPU. Nothing is published directly: the
Sahyog Setu server checks every translation (right script, numbers unchanged, not left in English, sensible
length) and fills **only empty** fields. Existing translations are never changed.

**Every time:**
1. Menu **Runtime → Change runtime type → T4 GPU → Save**.
2. Menu **Runtime → Run all**.
3. In step 2 paste your Hugging Face token (first time only: see the note in step 2).
4. Keep this tab open. If Colab stops (time limit), just open it again tomorrow and **Run all** — it continues where it stopped.


## 1. Settings

In [ ]:
JOB = "core"        # "core" first (quiz, menus, states, districts, documents, portals, scheme names ...), then "schemes"
LANGS = "all"       # or e.g. "ta,bn,te"
MODEL = "ai4bharat/indictrans2-en-indic-1B"   # the most accurate model (user: "language bilkul sahi ho")
BEAMS = 5           # 5 = best quality
BATCH = 24          # lower to 12 if you see "CUDA out of memory"
BASE = "https://sahyogsetu.in/i18n-job/__SAHYOG_TOKEN__"


## 2. Install + Hugging Face login
First time only: make a free account at **huggingface.co**, open
https://huggingface.co/ai4bharat/indictrans2-en-indic-1B and click **Agree / Access repository**, then
**Settings → Access Tokens → New token (Read)**, and paste that token below when asked.

In [ ]:
!pip -q install "transformers>=4.40,<4.47" sentencepiece sacremoses IndicTransToolkit 2>/dev/null || pip -q install git+https://github.com/VarunGumma/IndicTransToolkit.git
import torch, sys
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU, then Run all again"
print("GPU:", torch.cuda.get_device_name(0))
from huggingface_hub import login, HfFolder
if not HfFolder.get_token():
    login()


## 3. Load the model (2–4 minutes)

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
try:
    from IndicTransToolkit.processor import IndicProcessor
except ImportError:
    from IndicTransToolkit import IndicProcessor
tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL, trust_remote_code=True, torch_dtype=torch.float16).to("cuda").eval()
ip = IndicProcessor(inference=True)
print("model ready")


## 4. Translate (shows progress; safe to stop and run again later)

In [ ]:
import re, json, time, requests

def split_sentences(t):
    """Sentences of one segment (the model works best on one sentence at a time)."""
    parts = re.split(r'(?<=[.!?।])\s+(?=[A-Z0-9("\'])', t)
    out = []
    for p in parts:
        if len(p) <= 400:
            out.append(p); continue
        for q in re.split(r'(?<=[;:])\s+', p):           # very long sentence: split at ; or :
            out.append(q)
    return [p for p in out if p.strip()]

def translate(sents, tgt):
    res = [None] * len(sents)
    order = sorted(range(len(sents)), key=lambda i: len(sents[i]))
    for s in range(0, len(order), BATCH):
        idx = order[s:s + BATCH]
        batch = ip.preprocess_batch([sents[i] for i in idx], src_lang="eng_Latn", tgt_lang=tgt)
        enc = tok(batch, truncation=True, padding="longest", return_tensors="pt", return_attention_mask=True).to("cuda")
        with torch.no_grad():
            gen = model.generate(**enc, use_cache=True, min_length=0, max_length=256, num_beams=BEAMS, num_return_sequences=1)
        dec = tok.batch_decode(gen, skip_special_tokens=True, clean_up_tokenization_spaces=True)
        for i, t in zip(idx, ip.postprocess_batch(dec, lang=tgt)):
            res[i] = t
    return res

def get(url):
    for a in range(5):
        try:
            r = requests.get(url, timeout=60); r.raise_for_status(); return r.json()
        except Exception as e:
            print("  retry", a + 1, e); time.sleep(10 * (a + 1))
    raise SystemExit("server not reachable - run again later")

man = get(f"{BASE}/{JOB}/manifest.json")
langs = [l for l in man["langs"] if LANGS == "all" or l["code"] in LANGS.split(",")]
print(f"job {JOB}: {man['segments']:,} texts, {man['chunks']} batches, {len(langs)} languages")
done = get(f"{BASE}/{JOB}/done.json")
for L in langs:
    have = set(done.get(L["code"], []))
    todo = [n for n in range(man["chunks"]) if n not in have]
    if not todo:
        print(L["code"], "already complete"); continue
    print(f"== {L['code']} ({L['it2']}): {len(todo)} batches left")
    for n in todo:
        t0 = time.time()
        segs = get(f"{BASE}/{JOB}/chunk/{n}.json")
        flat, owner = [], []
        for j, it in enumerate(segs):
            for s in split_sentences(it["t"]):
                flat.append(s); owner.append(j)
        tr = translate(flat, L["it2"])
        joined = [[] for _ in segs]
        for j, t in zip(owner, tr):
            joined[j].append(t)
        payload = {it["k"]: " ".join(joined[j]).strip() for j, it in enumerate(segs) if joined[j]}
        for a in range(5):
            try:
                r = requests.post(f"{BASE}/{JOB}/result/{L['code']}/{n}", data=json.dumps(payload, ensure_ascii=False).encode("utf-8"),
                                  headers={"Content-Type": "application/json"}, timeout=120)
                r.raise_for_status(); break
            except Exception as e:
                print("  upload retry", a + 1, e); time.sleep(10 * (a + 1))
        print(f"   batch {n + 1}/{man['chunks']}: {len(segs)} texts in {time.time() - t0:.0f}s -> {r.json()}")
print("ALL DONE for", JOB)
